# Cardiac JEPA — campagne complète (Colab GPU)

Pipeline unique : pré-entraînement JEPA (ViT / CNN / xresnet) → sonde gelée + fine-tuning
(early-stopping) + référence supervisée BN, sur 5 graines, avec macro-AUROC + AUPRC + IC bootstrap.

**Idempotent** : chaque run dont le `result.json` (ou `best.pt`) existe déjà est sauté →
on relance après une coupure sans rien recalculer (runs sur le Drive).


## 0 · Configuration (à ajuster)


In [ ]:
REPO_URL   = 'https://github.com/JulesV19/cardiac-JEPA.git'
DRIVE_ROOT = '/content/drive/MyDrive/cardiac-jepa'   # runs/ + cache/ persistants ici
PTBXL_DIR  = f'{DRIVE_ROOT}/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1'  # dossier contenant ptbxl_database.csv + scp_statements.csv (+ cache/)
BRANCH     = 'main'


## 1 · Drive + repo + dépendances


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.isdir('/content/cardiac-jepa'):
    subprocess.run(['git','clone','-b',BRANCH,REPO_URL,'/content/cardiac-jepa'], check=True)
%cd /content/cardiac-jepa
!git pull --ff-only
!pip -q install wfdb tqdm


## 2 · Données (cache + métadonnées) + runs, sur le Drive

Le pipeline lit les **signaux** dans `cache/ptbxl_100hz.npy` (~1 Go) et les **labels/folds**
dans `ptbxl_database.csv` + `scp_statements.csv`. Les deux doivent être sur le Drive, sous
`PTBXL_DIR`. On les pointe par variables d'environnement (héritées par les `!python`).
La cellule échoue tôt et clairement si un fichier manque ou si le cache est désynchronisé.


In [ ]:
import os, numpy as np, pandas as pd
os.environ['PTBXL_DATA_DIR'] = PTBXL_DIR
os.environ['PTBXL_CACHE']    = f'{PTBXL_DIR}/cache/ptbxl_100hz.npy'  # ou f'{DRIVE_ROOT}/cache/...'

# runs/ persistant sur le Drive (poids + result.json y sont écrits)
runs_src = f'{DRIVE_ROOT}/runs'; os.makedirs(runs_src, exist_ok=True)
if not os.path.islink('runs'):
    import shutil
    if os.path.isdir('runs'): shutil.rmtree('runs')
    os.symlink(runs_src, 'runs')

# vérifs
csv = f'{PTBXL_DIR}/ptbxl_database.csv'
assert os.path.exists(csv), f'CSV PTB-XL introuvable: {csv}\n-> mets le dossier PTB-XL (au moins les 2 CSV) sur le Drive'
n = len(pd.read_csv(csv))
cache = os.environ['PTBXL_CACHE']
assert os.path.exists(cache), f'cache introuvable: {cache}\n-> copie ptbxl_100hz.npy sur le Drive, ou construis-le (voir cellule 2bis)'
rows = np.load(cache, mmap_mode='r').shape[0]
assert rows == n, f'cache desync: {rows} lignes vs CSV {n} -> reconstruire'
print(f'OK — CSV {n} ECG · cache {rows} lignes · runs -> {runs_src}')


### 2bis · (si le cache manque) le construire une fois

Nécessite les enregistrements bruts `records100/` sous `PTBXL_DIR`. À ne lancer qu'une fois.


In [ ]:
# !python -m jepa.build_cache   # écrit cache/ptbxl_100hz.npy (voir le module pour les options)


## 3 · Plan de la campagne (dry-run)

161 runs : 3 pré-entraînements + 18 sondes + 120 fine-tunings + 20 supervisés.


In [ ]:
!python -m jepa.experiment --dry-run | tail -3


## 4 · Étape A — pré-entraînements JEPA (les 3 backbones)

Le plus long (~100 epochs chacun). `--resume auto` reprend un pré-entraînement interrompu.


In [ ]:
!python -m jepa.experiment --steps pretrain


## 5 · Étape B — grille d'évaluation (sonde + fine-tuning + supervisé)

Relançable à volonté ; ne recalcule que ce qui manque. Trimmable : `--seeds 0,1,2`, `--backbones xresnet`.


In [ ]:
!python -m jepa.experiment --steps probe,finetune,supervised


## 6 · Agrégation → `runs/results.json`

Moyennes ± écart-type inter-graines, écarts pairés JEPA − scratch (t de Student).


In [ ]:
!python -m jepa.aggregate


## 7 · Empaqueter les artefacts LÉGERS pour les figures locales

Zippe `results.json` + tous les `result.json` + `metrics.csv` (PAS les poids `.pt`).
Quelques centaines de Ko. Décompresse à la racine du repo en local, puis `python make_figures.py`.


In [ ]:
import glob, zipfile
light = ['runs/results.json'] + glob.glob('runs/**/result.json', recursive=True) \
        + glob.glob('runs/**/metrics.csv', recursive=True)
light = [p for p in light if os.path.exists(p)]
with zipfile.ZipFile('results_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in light: z.write(p)
print(f'{len(light)} fichiers ->', round(os.path.getsize('results_bundle.zip')/1024), 'Ko')
from google.colab import files; files.download('results_bundle.zip')
